# v04 — Optimized Routing

End-to-end Colab runner for `v04_optimized_routing`. Same 2-pass pipeline as v03, but with an **optimized classification prompt** (domain-knowledge keywords, disambiguation rules, synthetic few-shot examples).

1. **Pass 1**: Classify each question into (domain, answer_type) using the optimized routing prompt
2. **Pass 2**: Solve with domain-specific system prompt + matched few-shot examples (same as v03)

See `ROUTING_PROMPT_DESIGN.md` for the methodology behind the prompt optimization.

All knobs (model, dtype, batch size, max tokens, ...) live in [`app/physics_solution/config.py`](../../config.py) — edit there, not here.

## 1. Mount Drive + chdir

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO_ROOT = '/content/drive/MyDrive/NCKH/Exact_2026_Laplace-s_Red_Devils'  # change if needed
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('cwd:', os.getcwd())

In [ ]:
!pip -q install -r app/physics_solution/requirements.txt

### Install Qwen3.5 fast-attention kernels (`flash-linear-attention` + `causal-conv1d`)

On Colab Oct-2025+ runtimes both wheels install cleanly. If `causal_conv1d: False` shows up, the model still runs via torch fallback at ~3× slower — accuracy is unaffected.

In [ ]:
from app.physics_solution.shared.colab.setup import install_fast_kernels
install_fast_kernels()

In [ ]:
from app.physics_solution.shared.colab.setup import setup_colab
setup_colab(repo_root=REPO_ROOT)

In [ ]:
!nvidia-smi -L

## 2. Choose test set

| File | Rows | Scope |
|---|---|---|
| `sample_test.csv` | 973 | Pure-numeric only (v01/v02 legacy) |
| `full_test.csv` | 1352 | **All answer types** (Stage 1+ baseline) |

Set `TEST_MODE` below to pick which one to run.

In [ ]:
import os

TEST_MODE = "full"  # "full" (1352 rows, all types) or "sample" (973, pure-numeric)

if TEST_MODE == "full":
    TEST_FILE = 'app/physics_solution/data/test/full_test.csv'
    OUT_FILE  = 'app/physics_solution/versions/v04_optimized_routing/output/results_full.json'
    if not os.path.exists(TEST_FILE):
        !python -m app.physics_solution.cli.prepare_sample \
            --n 1352 --seed 42 --answer-types all --out {TEST_FILE}
    else:
        print('test set exists:', TEST_FILE)
else:
    TEST_FILE = 'app/physics_solution/data/test/sample_test.csv'
    OUT_FILE  = 'app/physics_solution/versions/v04_optimized_routing/output/results.json'
    if not os.path.exists(TEST_FILE):
        !python -m app.physics_solution.cli.prepare_sample \
            --n 973 --seed 42 --out {TEST_FILE}
    else:
        print('test set exists:', TEST_FILE)

print(f'\n\u2192 Using: {TEST_FILE}  \u2192  {OUT_FILE}')

## 3. Curate few-shot examples

v04 shares the same few-shot pool as v03. If you haven't run v03's `select_fewshot` yet, run it below.
The v04 runner will automatically fall back to v03's `examples.json` if v04 doesn't have its own.

In [ ]:
import os
v03_examples = 'app/physics_solution/versions/v03_routed_fewshot/input/examples.json'
v04_examples = 'app/physics_solution/versions/v04_optimized_routing/input/examples.json'

if os.path.exists(v04_examples):
    print('v04 examples exist:', v04_examples)
elif os.path.exists(v03_examples):
    print('v03 examples exist (v04 will use these as fallback):', v03_examples)
else:
    print('No examples found. Running v03 select_fewshot...')
    !python -m app.physics_solution.versions.v03_routed_fewshot.select_fewshot --k 2 --seed 42

## 4. Inference (2-pass: classify → route → solve)

All knobs (batch size, dtype, max_new_tokens, ...) come from `config.py`. Add e.g. `--limit 50 --batch-size 8` to override per-run for smoke tests.

In [ ]:
!python -m app.physics_solution.cli.inference \
    --version v04_optimized_routing \
    --out {OUT_FILE}

## 5. Push to HF (with full metadata)

In [ ]:
!python -m app.physics_solution.cli.upload_model \
    --version-num 4 --strategy optimized_routing \
    --results {OUT_FILE}

## 6. Inspect results

In [ ]:
import json, pandas as pd
data = json.loads(open(OUT_FILE, encoding='utf-8').read())
print('summary:', json.dumps(data['summary'], indent=2, ensure_ascii=False))
df = pd.DataFrame(data['rows'])
wrong = df[~df['is_correct']].copy()
wrong['reached_final'] = wrong['completion'].str.contains('FINAL ANSWER', case=False, regex=False)
print(f"\nWrong: {len(wrong)} / {len(df)}")
print(f"  ... never reached FINAL ANSWER: {(~wrong['reached_final']).sum()}")
wrong[['id', 'gold_answer', 'pred_numeric', 'pred_unit', 'gold_unit', 'reached_final']].head(20)

## 7. Routing accuracy analysis

Compare predicted domain (from Pass 1 classifier) vs the ID prefix ground truth.
This is the key metric for evaluating the optimized routing prompt.

In [ ]:
import re
PREFIX_RE = re.compile(r"^([A-Z]+)")

# Extract routing info from extra column
df['routed_domain'] = df['extra'].apply(lambda x: x.get('routed_domain', '') if isinstance(x, dict) else '')
df['routed_answer_type'] = df['extra'].apply(lambda x: x.get('routed_answer_type', '') if isinstance(x, dict) else '')
df['true_domain'] = df['id'].apply(lambda x: PREFIX_RE.match(str(x)).group(1) if PREFIX_RE.match(str(x)) else '')

df['domain_correct'] = df['routed_domain'] == df['true_domain']
print(f"Domain routing accuracy: {df['domain_correct'].sum()}/{len(df)} = {df['domain_correct'].mean():.3f}")

# Per-domain routing accuracy
print("\nPer-domain routing accuracy:")
for domain, sub in df.groupby('true_domain'):
    acc = sub['domain_correct'].mean()
    print(f"  {domain}: {sub['domain_correct'].sum()}/{len(sub)} = {acc:.3f}")

# Confusion: what domains were misclassified as what
misrouted = df[~df['domain_correct']]
if len(misrouted) > 0:
    print(f"\nMisrouted examples ({len(misrouted)} total):")
    print(pd.crosstab(misrouted['true_domain'], misrouted['routed_domain'], margins=True))

## 8. v03 vs v04 comparison

If you have v03 results available, compare routing + solving accuracy side by side.

In [ ]:
import os
v03_results_path = 'app/physics_solution/versions/v03_routed_fewshot/output/results.json'

if os.path.exists(v03_results_path):
    v03_data = json.loads(open(v03_results_path, encoding='utf-8').read())
    v03_summary = v03_data['summary']
    v04_summary = data['summary']
    print('=== v03 vs v04 Comparison ===')
    print(f"  v03 accuracy: {v03_summary['correct']}/{v03_summary['n']} = {v03_summary['accuracy']:.3f}")
    print(f"  v04 accuracy: {v04_summary['correct']}/{v04_summary['n']} = {v04_summary['accuracy']:.3f}")
    print(f"  Delta: {v04_summary['accuracy'] - v03_summary['accuracy']:+.3f}")
    print(f"\n  v03 mean latency: {v03_summary['mean_latency_s']:.2f}s")
    print(f"  v04 mean latency: {v04_summary['mean_latency_s']:.2f}s")
else:
    print(f'v03 results not found at {v03_results_path} — run v03 first for comparison.')

## 9. Deep error analysis

Open [`app/physics_solution/eda/notebooks/error_analysis.ipynb`](../../eda/notebooks/error_analysis.ipynb) — it categorises every wrong row into fail modes and writes a markdown report.